In [8]:
import ROOT
import pandas as pd
import numpy as np
import os
import sys
import ipynbname
from pathlib import Path

project_root = str(ipynbname.path().parent.parent.parent)
sys.path.append(project_root)
project_root=Path(project_root)

from processing import SiphraAcquisition, MetadataLoader
from analysis import fit_peak_expbg

# Constants
BITS12 = 2**12
BITS9 = 2**9 # 512 typical number of bins used
# Energy emission peaks in MeV
K40_MEV = [1.460]
TL208_MEV = [2.614]
CS137_MEV = [0.661]
ANNIHIL_MEV = 0.511
NA22_MEV = [ANNIHIL_MEV, 1.2745, 1.2745+ANNIHIL_MEV]
CO60_MEV = [(1.1732+1.3325)/2, 1.1732+1.3325] # First value is the average of the two photopeaks. They have the same branching ratio
CO60_MEV_3PEAKS = [1.1732, 1.3325, 1.1732+1.3325]

# colors = [ROOT.kRed, ROOT.kBlue, ROOT.kGreen, ROOT.kOrange, ROOT.kAzure, ROOT.kSpring, ROOT.kPink, ROOT.kMagenta, ROOT.kTeal, ROOT.kYellow, ROOT.kViolet, ROOT.kCyan]
# colors = [ROOT.kAzure+9, ROOT.kOrange+1, ROOT.kRed+1, ROOT.kGray+1, ROOT.kViolet-5, ROOT.kRed-5, ROOT.kOrange+8, ROOT.kYellow-5, ROOT.kGray+2, ROOT.kCyan-8]
colors = [ROOT.kAzure-7, ROOT.kOrange+1, ROOT.kRed-7, ROOT.kCyan-5, ROOT.kGreen-5, ROOT.kOrange-4, ROOT.kMagenta-5, ROOT.kRed-9, ROOT.kOrange-7, ROOT.kGray+1]

In [9]:
file_nrs = [15, 16, 18, 20]
files_A = sorted([list((project_root/'data/260603').glob(f'SUBT{nr}_*.csv'))[0] for nr in file_nrs])
# files_B = sorted((project_root/'data/260519').glob('SUBT_B0[1-4]*.csv'))
print(files_A)
# print(files_B)


[PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260603/SUBT15_LongBackgroundAtEnd.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260603/SUBT16_Cs137OnCrystalAtCenter.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260603/SUBT18_Co60OnCrystalAtCenter.csv'), PosixPath('/home/oscar/Projects/SIPHRA_Testing/data/260603/SUBT20_Na22OnCrystalAtCenter.csv')]


In [10]:
acqs_A = [SiphraAcquisition(file) for file in files_A]
# acqs_B = [SiphraAcquisition(file) for file in files_B]
print(acqs_A)
# print(acqs_B)

[SIPHRAAcquisition(File: 'SUBT15_LongBackgroundAtEnd'), SIPHRAAcquisition(File: 'SUBT16_Cs137OnCrystalAtCenter'), SIPHRAAcquisition(File: 'SUBT18_Co60OnCrystalAtCenter'), SIPHRAAcquisition(File: 'SUBT20_Na22OnCrystalAtCenter')]


In [12]:
# histograms
from collections import namedtuple

DataHandler = namedtuple('DataHandler', ['acq', 'hist', 'src', 'subt'])

hists = {}
sources = []
handlers = {}
for acq in acqs_A:
    filepath = acq.filepath.stem
    src = (MetadataLoader.load(acq.metadataFile)).source
    src_key = 'BG' if src == 'Background' else src[:2]
    sources.append(src)
    # siphra_code = acq.filepath.stem[5]
    print(src)
    # Generate histograms
    hist = ROOT.TH1F(f"{src}", "", BITS12, 0, BITS12)
    hist.Fill(acq['s']/len(acq.active_chs))
    hists[f'{src_key}'] = hist
    handlers[f'{src_key}'] = DataHandler(acq, hist, src, None)
    del hist

Background
Cs-137
Co-60
Na-22


Warning in <TROOT::Append>: Replacing existing TH1: Background (Potential memory leak).


In [13]:
handlers

{'BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT15_LongBackgroundAtEnd'), hist=<cppyy.gbl.TH1F object at 0x617164ce4a40>, src='Background', subt=None),
 'Cs': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT16_Cs137OnCrystalAtCenter'), hist=<cppyy.gbl.TH1F object at 0x617164d37ef0>, src='Cs-137', subt=None),
 'Co': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT18_Co60OnCrystalAtCenter'), hist=<cppyy.gbl.TH1F object at 0x617164ce1c80>, src='Co-60', subt=None),
 'Na': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT20_Na22OnCrystalAtCenter'), hist=<cppyy.gbl.TH1F object at 0x617164ce23a0>, src='Na-22', subt=None)}

In [15]:
correction_factors = {
    'Cs': 1.8,
    'Na': 1.7,
    'Co': 1.8,
}

In [87]:
# Subtract backgrounds here if correction factors are not configured, checking each one

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgd = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

whichSiphra = 'A'
source = 'Co'
bg = handlers[f'BG']
data_key = f'{source}'
sgnl = handlers[data_key]

bg_scaled = bg.hist.Clone()
correction_factor = correction_factors[data_key] if correction_factors[data_key] else 1
bg_scaled.Scale(correction_factor*sgnl.acq.exposure/bg.acq.exposure)

subtracted = sgnl.hist.Clone(f'{sgnl.src}_{whichSiphra}_Subt.')
subtracted.Add(bg_scaled, -1)
# subts[data_key].hist = subtracted

lgd.AddEntry(sgnl.hist, 'Signal')
lgd.AddEntry(bg_scaled, 'Background')
lgd.AddEntry(subtracted, 'Subtracted')
sgnl.hist.SetLineColor(colors[4])
bg_scaled.SetLineColor(colors[3])
subtracted.SetLineColor(colors[2])
sgnl.hist.SetTitle(f'{sgnl.src}')
sgnl.hist.Draw('hist')
bg_scaled.Draw('hist sames')
subtracted.Draw('hist sames')
cv.SetLogy()
lgd.Draw()
cv.Draw()

print(f'{sgnl.acq.exposure=}, {bg.acq.exposure=}, {correction_factor=}')

new_handler = DataHandler(sgnl.acq, sgnl.hist, sgnl.src, subtracted)
handlers[data_key] = new_handler

sgnl.acq.exposure=229.02611804008484, bg.acq.exposure=459.8542814254761, correction_factor=1.8


In [20]:
# Execute here if compensation factors are established
whichSiphra = 'A'
bg = handlers[f'BG']
for source in ['Cs', 'Na', 'Co']:
    data_key = f'{source}'
    sgnl = handlers[data_key]
    
    bg_scaled = bg.hist.Clone(f'bg_scaled_{source}')
    correction_factor = correction_factors[data_key] if correction_factors[data_key] else 1
    bg_scaled.Scale(correction_factor*sgnl.acq.exposure/bg.acq.exposure)
    
    subtracted = sgnl.hist.Clone(f'{sgnl.src}_{whichSiphra}_Subt.')
    subtracted.Add(bg_scaled, -1)
    
    print(f'{sgnl.acq.exposure=}, {bg.acq.exposure=}, {correction_factor=}')
    
    new_handler = DataHandler(sgnl.acq, sgnl.hist, sgnl.src, subtracted)
    handlers[data_key] = new_handler

sgnl.acq.exposure=217.42797112464905, bg.acq.exposure=459.8542814254761, correction_factor=1.8
sgnl.acq.exposure=224.20187497138977, bg.acq.exposure=459.8542814254761, correction_factor=1.7
sgnl.acq.exposure=229.02611804008484, bg.acq.exposure=459.8542814254761, correction_factor=1.8


In [21]:
handlers

{'BG': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT15_LongBackgroundAtEnd'), hist=<cppyy.gbl.TH1F object at 0x617164ce4a40>, src='Background', subt=None),
 'Cs': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT16_Cs137OnCrystalAtCenter'), hist=<cppyy.gbl.TH1F object at 0x617164d37ef0>, src='Cs-137', subt=<cppyy.gbl.TH1F object at 0x617163235740>),
 'Co': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT18_Co60OnCrystalAtCenter'), hist=<cppyy.gbl.TH1F object at 0x617164ce1c80>, src='Co-60', subt=<cppyy.gbl.TH1F object at 0x6171664b4810>),
 'Na': DataHandler(acq=SIPHRAAcquisition(File: 'SUBT20_Na22OnCrystalAtCenter'), hist=<cppyy.gbl.TH1F object at 0x617164ce23a0>, src='Na-22', subt=<cppyy.gbl.TH1F object at 0x61716a9de3f0>)}

In [22]:
if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

Yinit = 0.82 # For stat boxes

cv = ROOT.TCanvas('cv', 'cv', 1200, 800)

ROOT.gStyle.SetOptStat(11)
ROOT.gStyle.SetStatFontSize(0.03)
ROOT.gStyle.SetStatW(0.16)

lgds = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

srcs_list = list(handlers.values())
srcs_list.pop(0)

for n, handler in enumerate(srcs_list):
    hist = handler.subt
    lgds.AddEntry(hist, handler.src)
    if not n%3:
        s = 'A' if n < 3 else 'B'
        hist.SetTitle(f'Detector {s}')
        hist.GetXaxis().SetTitle("ADC channel number")
        hist.GetYaxis().SetTitle("Relative counts")
        hist.SetLineColor(colors[(n%3)])
        hist.Draw('hist')
    else:
        hist.SetLineColor(colors[(n%3)])
        hist.Draw('hist sames')

for c in range(2):
    cv.cd(c+1)
    lgds.Draw()
    cv.cd(c+1).SetLogy()

cv.Draw()


In [23]:
# Fit ranges for each peak. Same order as in srcs_list
fit_ranges = {
    'Cs': [(368,588)], # Cs siphra A
    'Na': [(257, 464),(756,927),(1041, 1287)], # Na siphra A
    'Co': [(671, 950),(1452, 1773)], # Co  - Fit first range to a double gaussian
}

for k, ranges in fit_ranges.items():
    try:
        hist = handlers[k].subt
        source = handlers[k].src
        chip = k[0]
        # print(f'{k}: chip: {chip}, src: {source}')
    except:
        print(f'Unmatched key {k}')

    for n, r in enumerate(ranges):
        print(f'{source=}, {n=}')
        if source == 'Co-60' and n == 0:
            print("Fitting Cobalt double peak")
            fit_fn = ROOT.TF1(f"{source}_fit_{n+1}", "gaus(0) + gaus(3)", 6)
            fit_fn.SetParameter(0, 83)  # Constant
            fit_fn.SetParLimits(0, 50, 80)
            fit_fn.SetParameter(1, 738) # Mean
            fit_fn.SetParLimits(1, 700, 800)
            fit_fn.SetParameter(2,158)  # Sigma
            fit_fn.SetParLimits(2, 0, 500)
            fit_fn.SetParameter(3, 63)
            fit_fn.SetParLimits(3, 50, 80)
            fit_fn.SetParameter(4, 882)
            fit_fn.SetParLimits(4, 800, 900)
            fit_fn.SetParameter(5,158)
            fit_fn.SetParLimits(5, 0, 500)
        else:
            fit_fn = ROOT.TF1(f"{source}_fit_{n+1}", "gaus", 3)
        fit_status = hist.Fit(fit_fn, 'L S +', "", r[0], r[1])

source='Cs-137', n=0
source='Na-22', n=0
source='Na-22', n=1
source='Na-22', n=2
source='Co-60', n=0
Fitting Cobalt double peak
source='Co-60', n=1
****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      458.458
Chi2                      =      916.917
NDf                       =          217
Edm                       =  5.52595e-06
NCalls                    =           67
Constant                  =      3351.01   +/-   7.33578     
Mean                      =      459.209   +/-   0.0724202   
Sigma                     =      39.6441   +/-   0.0568644    	 (limited)
****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      322.377
Chi2                      =      644.755
NDf                       =          204
Edm                       =  1.02569e-07
NCalls                    =           62
Constant                  =      215.641   +/-   1.86073     
Mean                      =      366.033

Info in <ROOT::Math::ParameterSettings>: lower/upper bounds outside current parameter value. The value will be set to (low+up)/2 


In [36]:
# graph of position of photopeaks

def get_mean_and_error(func_name, hist):
    f = hist.GetFunction(func_name)
    idx = f.GetParNumber('Mean')
    return f.GetParameter(idx), f.GetParError(idx)

def get_par_and_error(func_name, hist, par_nr):
    f = hist.GetFunction(func_name)
    return f.GetParameter(par_nr), f.GetParError(par_nr)

COLOR_A = colors[0]
COLOR_B = colors[1]

means_A, errors_A = zip(*[
    get_mean_and_error('Cs-137_fit_1', handlers['Cs'].subt),
    get_mean_and_error('Na-22_fit_1',  handlers['Na'].subt),
    get_mean_and_error('Na-22_fit_2',  handlers['Na'].subt),
    get_mean_and_error('Na-22_fit_3',  handlers['Na'].subt),
    get_par_and_error('Co-60_fit_1',  handlers['Co'].subt, 1),
    get_par_and_error('Co-60_fit_1', handlers['Co'].subt, 4),
    get_mean_and_error('Co-60_fit_2',  handlers['Co'].subt),
])

sigmas_A, errs_sig_A = zip(*[
    get_par_and_error('Cs-137_fit_1', handlers['Cs'].subt, 2),
    get_par_and_error('Na-22_fit_1',  handlers['Na'].subt, 2),
    get_par_and_error('Na-22_fit_2',  handlers['Na'].subt, 2),
    get_par_and_error('Na-22_fit_3',  handlers['Na'].subt, 2),
    get_par_and_error('Co-60_fit_1',  handlers['Co'].subt, 2),
    get_par_and_error('Co-60_fit_1', handlers['Co'].subt, 5),
    get_par_and_error('Co-60_fit_2',  handlers['Co'].subt, 2),
])

means_A  = np.array(means_A,  dtype=np.float64)
errors_A = np.array(sigmas_A, dtype=np.float64)

peaks_real = np.array([*CS137_MEV, *NA22_MEV, *CO60_MEV_3PEAKS])

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)
lg = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

errs_values = np.array([1e-6, 1e-8, 1e-7, 1e-7, 1e-7, 1e-7, 1e-7])

peaks_graph_A = ROOT.TGraphErrors(7, means_A, peaks_real, errors_A, errs_values)
peaks_graph_A.SetMarkerColor(COLOR_B)
peaks_graph_A.SetMarkerStyle(20)

peaks_graph_A.SetTitle("Position of the photopeaks of the radiation sources")
peaks_graph_A.GetXaxis().SetTitle("ADC channel number")
peaks_graph_A.GetYaxis().SetTitle("Energy (MeV)")

lg.AddEntry(peaks_graph_A, "Detector A")
# lg.AddEntry(peaks_graph_B, "SIPHRA B")
peaks_graph_A.Draw("AP")
# peaks_graph_B.Draw("P same")
lg.Draw()
cv.Draw()

In [27]:
np.sqrt(1e-8**2 + 1e-7**2)

np.float64(1.0049875621120889e-07)

In [32]:
# Fitting the calibration curves

calib_A = ROOT.TF1('Calibration_A', 'pol1', means_A.min()-200, means_A.max()+200)
peaks_graph_A.Fit(calib_A, 'RSF 0')

# calib_B = ROOT.TF1('Calibration_B', 'pol1', means_B.min()-50, means_B.max()+50)
# peaks_graph_B.Fit(calib_B, 'RS 0')



****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =     0.381666
NDf                       =            5
Edm                       =  4.44532e-07
NCalls                    =           94
p0                        =   -0.0636525   +/-   0.072045    
p1                        =    0.0015952   +/-   9.31617e-05 


In [68]:
#### if ROOT.gROOT.FindObject('cv'):
ROOT.gROOT.FindObject('cv').Close()

labels = [
    "{}^{137}Cs " + f"{CS137_MEV[0]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[0]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[1]:.2f} MeV",
    "{}^{22}Na " + f"{NA22_MEV[2]:.2f} MeV",
    "{}^{60}Co " + f"{CO60_MEV_3PEAKS[0]:.2f} MeV",
    "{}^{60}Co " + f"{CO60_MEV_3PEAKS[1]:.2f} MeV",
    "{}^{60}Co " + f"{CO60_MEV_3PEAKS[2]:.2f} MeV",
    "{}^{40}K "  + f"{K40_MEV[0]:.2f} MeV",
    "{}^{208}Tl "+ f"{TL208_MEV[0]:.2f} MeV",
          ]

real_peaks_ordered = [*CS137_MEV, *NA22_MEV, *CO60_MEV_3PEAKS, *K40_MEV, *TL208_MEV]


cv = ROOT.TCanvas('cv', 'cv', 1200, 800)
lg = ROOT.TLegend(0.15, 0.75, 0.36, 0.88)

lg.AddEntry(peaks_graph_A, "Detector A")
# lg.AddEntry(peaks_graph_B, "Detector B")

calib_A.SetLineColor(COLOR_A)
calib_A.SetLineStyle(2)
# calib_B.SetLineColor(COLOR_B)
# calib_B.SetLineStyle(2)

peaks_graph_A.Draw("AP")
# peaks_graph_B.Draw("P same")
calib_A.Draw("same")
lg.AddEntry(calib_A, "Linear fit")
# calib_B.Draw("same")
lg.Draw()

# Fit parameters

# eqbox_A = ROOT.TPaveText(0.15, 0.65, 0.45, 0.85, "NDC")
# eqbox_A.SetFIllColor(0)
# eqbox_A.(1)

# get parameters
calibPars = {}
eqboxes = {}
for chip, fitfn in zip(['A',], [calib_A,]):
    eqboxes[f'{chip}'] = ROOT.TPaveText(0.15, 0.7, 0.48, 0.8, "NDC")
    box =  eqboxes[f'{chip}']
    box.SetFillColor(0)
    box.SetBorderSize(1)
    a = fitfn.GetParameter(0)
    b = fitfn.GetParameter(1)
    sa = fitfn.GetParError(0)
    sb = fitfn.GetParError(1)

    calibPars[f'a_{chip}'] = a
    calibPars[f'b_{chip}'] = b
    calibPars[f'sa_{chip}'] = sa
    calibPars[f'sb_{chip}'] = sb

    box.AddText(f"Detector {chip}")
    box.AddText(f"E = ({a:.4f} #pm {sa:.4f}) + ({b:.4f} #pm {sb:.4f}) x")
    box.Draw()

# Add labels
latex = ROOT.TLatex()
latex.SetTextSize(0.02)
latex.SetTextAlign(12)  # alineación: horizontal izquierda, vertical centrada

for i, adc in enumerate(means_A):
    latex.DrawLatex(adc+ 0.05, real_peaks_ordered[i], labels[i])  # pequeño offset en x


cv.Update()
cv.Draw()

# Dynamic range and verification

In [61]:
# Calibrate BG spectra

a = calibPars['a_A']
b = calibPars['b_A']
sa = calibPars['sa_A']
sb = calibPars['sb_A']

def calibrate(data, return_errs=False):
    ''' Performs linear energy calibration (a + bx) over a data array containing ADC values.'''
    value = a + b*data
    if return_errs:
        dva = 1
        dvb = b
        err = np.sqrt((dva*sa)**2 + (dvb*sb)**2)
        return a + b*data, err
    return value

data_bgA_calib = calibrate(handlers['BG'].acq['s']/16)
# data_bgB_calib = calibrate(handlers['B_BG'].acq['s']/16, calibPars['a_B'], calibPars['b_B'])

hist_bgA_calib = ROOT.TH1F(f"BG_SIPHRA_A", "", BITS12, data_bgA_calib.min(), data_bgA_calib.max())
hist_bgA_calib.Fill(data_bgA_calib)
# hist_bgB_calib = ROOT.TH1F(f"BG_SIPHRA_B", "", BITS12, data_bgB_calib.min(), data_bgB_calib.max())
# hist_bgB_calib.Fill(data_bgB_calib)

Warning in <TROOT::Append>: Replacing existing TH1: BG_SIPHRA_A (Potential memory leak).


In [49]:
print('Dynamic ranges:')
min_bin = -calibPars['a_A']/calibPars['b_A']
pedestal_max = 150
pedestal_energy = calibrate(pedestal_max, return_errs=True)
dynRange_min = calibrate(min_bin, return_errs=True)
dynRange_max = calibrate(BITS12, return_errs=True)
print(f'Detector A: [({dynRange_min[0]:.4f} ± {dynRange_min[1]}) - ({dynRange_max[0]:.4f} ± {dynRange_max[1]:.4f})] MeV')
# print(f'SIPHRA B: {(calibPars['a_B'] + calibPars['b_B']*BITS12)-(calibPars['a_A'] + calibPars['b_A']*0.05):.4f} MeV')
print(f'Pedestal: {pedestal_max}ADC = {pedestal_energy}MeV, {min_bin=} ')

Dynamic ranges:
Detector A: [(0.0000 ± 0.07204500628359893) - (6.4703 ± 0.0720)] MeV
Pedestal: 150ADC = (0.17562685434197772, np.float64(0.07204500628359893))MeV, min_bin=39.9026429253971 


In [63]:
import visualization

visualization.hist_quickShow([hist_bgA_calib], legends=['Detector A'], color_offset=1, tone_offset=0)

(<cppyy.gbl.TCanvas object at 0x61716ec51a40>,
 <cppyy.gbl.TLegend object at 0x61716b57cc20>)

In [62]:
import importlib
if 'analysis' in sys.modules:
    del sys.modules['analysis']

from analysis import fit_peak_expbg

print(f'{K40_MEV[0]=}    {TL208_MEV[0]=}')

fit_peak_expbg(hist_bgA_calib, 'K_peak', 1.3, 1.7, 28, 1.5, 0.3, showFit=True, keep_prev_fncs=True)
fit_peak_expbg(hist_bgA_calib, 'Tl_peak', 2.55, 2.7, 41, 4, 0.3, showFit=True, keep_prev_fncs=True)

# fit_peak_expbg(hist_bgB_calib, 'K_peak', 1.15, 2, 315, 1.5, 0.3, showFit=True, keep_prev_fncs=True)
# fit_peak_expbg(hist_bgB_calib, 'Tl_peak', 2.25, 2.75, 41, 2.6, 0.3, showFit=True, keep_prev_fncs=True)

K40_MEV[0]=1.46    TL208_MEV[0]=2.614


(<cppyy.gbl.TF1 object at 0x61716ec51880>,
 <cppyy.gbl.TFitResultPtr object at 0x61716b69e490>)

****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      131.732
Chi2                      =      263.463
NDf                       =          255
Edm                       =   3.4199e-06
NCalls                    =          778
Const                     =      2.24018   +/-   0.665893    
Decay                     =    -0.599703   +/-   0.381185    
Norm                      =      14.7807   +/-   0.898       
Mean                      =      1.44207   +/-   0.00447045  
Sigma                     =     0.106673   +/-   0.0079861   
****************************************
Minimizer is Minuit2 / Migrad
MinFCN                    =      54.9969
Chi2                      =      109.994
NDf                       =           92
Edm                       =  2.21057e-09
NCalls                    =          971
Const                     =     -8.42131   +/-   0.930695    
Decay                     =    -0.868282   +/-   5.68747     
Norm          

In [64]:
bgHists = [hist_bgA_calib, ]
bgPeaks_real = [K40_MEV, TL208_MEV]
kpeaks = [bgHists[0].GetFunction('K_peak').GetParameter('Mean'), ]
tlpeaks = [bgHists[0].GetFunction('Tl_peak').GetParameter('Mean'), ]
errs = []
errs = [[(expr - real[0])/real[0] for expr in l_exp] for real, l_exp in zip(bgPeaks_real, [kpeaks, tlpeaks])]
print("Errors in the energy reconstruction of the background K-40 and Tl-208 photopeaks")
print(f'{'-'*50}')
print(f'{'Source':^10} | {'SIPHRA A':^15} | {'SIPHRA B':^15}')
print(f'{'-'*50}')
print(f'{'K-40':^10} | {f'{errs[0][0]*100:.3f} %':>15} | 0')
print(f'{'Tl-208':^10} | {f'{errs[1][0]*100:.3f} %':>15} | 0')
print(f'{'-'*50}')

Errors in the energy reconstruction of the background K-40 and Tl-208 photopeaks
--------------------------------------------------
  Source   |    SIPHRA A     |    SIPHRA B    
--------------------------------------------------
   K-40    |        -1.228 % | 0
  Tl-208   |         1.640 % | 0
--------------------------------------------------


# Energy resolution

For the energy resolution, we can get the calibrated data from the fitted data of the spectra we previously obtained. For a linear calibration $E = a + b \cdot x$, the transform for the parameters of the fitted gaussians is:

**Mean**

$\mu_E = a + b\cdot \mu_x$

**Sigma**

$\sigma_E = b \cdot \sigma_x$

The propagated error in the sigma is

$\sigma^2_{\sigma_E} = b^2\sigma^2_{\sigma_x} + \sigma^2_x\sigma^2_b + 2b\sigma_x \mathrm{cov}(\sigma_x, b)$

The last term is negligible since $\sigma_x$ comes from the histogram fit and $b$ from the calibration, so they are independent.

In [65]:
def get_sigma_and_error(func_name, hist):
    f = hist.GetFunction(func_name)
    idx = f.GetParNumber('Sigma')
    return f.GetParameter(idx), f.GetParError(idx)

COLOR_A = colors[0]
COLOR_B = colors[1]



# sigmas_A, errs_sig_A = zip(*[
#     get_sigma_and_error('Cs-137_gaus_1', handlers['Cs'].subt),
#     get_sigma_and_error('Na-22_gaus_1',  handlers['Na'].subt),
#     get_sigma_and_error('Na-22_gaus_2',  handlers['Na'].subt),
#     get_sigma_and_error('Na-22_gaus_3',  handlers['Na'].subt),
#     get_sigma_and_error('Co-60_gaus_1',  handlers['Co'].subt),
#     get_sigma_and_error('Co-60_gaus_2',  handlers['Co'].subt),
# ])

# sigmas_B, errs_sig_B = zip(*[
#     get_sigma_and_error('Cs-137_gaus_1', handlers['B_Cs'].subt),
#     get_sigma_and_error('Na-22_gaus_1',  handlers['B_Na'].subt),
#     get_sigma_and_error('Na-22_gaus_2',  handlers['B_Na'].subt),
#     get_sigma_and_error('Na-22_gaus_3',  handlers['B_Na'].subt),
#     get_sigma_and_error('Co-60_gaus_1',  handlers['B_Co'].subt),
#     get_sigma_and_error('Co-60_gaus_2',  handlers['B_Co'].subt),
# ])

PeakPars = namedtuple('PeakPars', ['mean', 'sigma', 'err_mean', 'err_sigma', 'fwhm', 'res'])

fwhm_factor = np.sqrt(8*np.log(2))
src_peaks_pars = {}
peak_names = ['Cs1', 'Na1', 'Na2', 'Na3', 'Co1', 'Co2', 'Co3']
real_peaks_ordered = [*CS137_MEV, *NA22_MEV, *CO60_MEV_3PEAKS, *K40_MEV, *TL208_MEV]

for chip, means, err_means, sigs, err_sigs, a, b, err_b in zip(('A',), (means_A,), (errors_A,), (sigmas_A,), (errs_sig_A,), (calibPars['a_A'], ), (calibPars['b_A'], ), (calibPars['sb_A'], )):
    # print(chip, means, err_means, sigs, err_sigs, a, b, err_b, '\n')
    for name, m, sm, s, ss, r in zip((peak_names), means, err_means, sigs, err_sigs, real_peaks_ordered):
        mean, err_mean = a + b*m, np.sqrt((b*sm)**2 + (m*err_b)**2)
        sig, err_sig = b*s, np.sqrt((b*ss)**2 + (s*err_b)**2)
        fwhm = fwhm_factor*sig
        res = fwhm/mean
        src_peaks_pars[f'{chip}_{name}'] = PeakPars(mean=mean, sigma=sig, err_mean=err_mean, err_sigma=err_sig, fwhm=fwhm, res=res*100)
        
print(real_peaks_ordered)

for k, v in src_peaks_pars.items():
    print(f'{k}:\n {v}\n')

[0.661, 0.511, 1.2745, 1.7854999999999999, 1.1732, 1.3325, 2.5057, 1.46, 2.614]
A_Cs1:
 PeakPars(mean=np.float64(0.6688764539431095), sigma=0.06324007693542179, err_mean=np.float64(0.07635113768446751), err_sigma=np.float64(0.003694421826707893), fwhm=np.float64(0.14891900081683063), res=np.float64(22.264051894626384))

A_Na1:
 PeakPars(mean=np.float64(0.5202418072244869), sigma=0.06424307525229807, err_mean=np.float64(0.07273237890830271), err_sigma=np.float64(0.003770023322688696), fwhm=np.float64(0.1512808813585432), res=np.float64(29.078955066228417))

A_Na2:
 PeakPars(mean=np.float64(1.2795698262711055), sigma=0.0922104630601162, err_mean=np.float64(0.12106424555908869), err_sigma=np.float64(0.005960095312730016), fwhm=np.float64(0.21713904677554752), res=np.float64(16.96969108816276))

A_Na3:
 PeakPars(mean=np.float64(1.810474232746915), sigma=0.10994888838422388, err_mean=np.float64(0.15513997014789893), err_sigma=np.float64(0.006677079029282527), fwhm=np.float64(0.2589098462960

In [66]:
resolutions = {'A':[],}

for k, v in src_peaks_pars.items():
    resolutions[k[0]].append(v.res)

# Add the background peaks data
real_peaks_bg = [K40_MEV[0], TL208_MEV[0]]
res_bg = {'A': [hist_bgA_calib.GetFunction(fnc).GetParameter('Sigma') for fnc in ['K_peak', 'Tl_peak']],}
# print(res_bg)

means_bg = {'A':[hist_bgA_calib.GetFunction(fnc).GetParameter('Mean') for fnc in ['K_peak', 'Tl_peak']],}

for k, v in res_bg.items():
    # print(k, v)
    res_bg[k] = [fwhm_factor*100*sig/r for sig, r in zip(v, means_bg[k])]

resolutions['A'].extend(res_bg['A'])
# resolutions['B'].extend(res_bg['B'])

print(f'{resolutions=}')

resolutions={'A': [np.float64(22.264051894626384), np.float64(29.078955066228417), np.float64(16.96969108816276), np.float64(14.300664522754008), np.float64(18.154491008625293), np.float64(15.62621078511609), np.float64(12.297846987787212), np.float64(17.41914791435846), np.float64(9.495406760696927)]}


In [69]:
real_peaks_ordered = np.array(real_peaks_ordered)
resolutions['A'] = np.array(resolutions['A'])
# resolutions['B'] = np.array(resolutions['B'])

# labels = [
#     "{}^{137}Cs " + f"{CS137_MEV[0]:.2f} MeV",
#     "{}^{22}Na " + f"{NA22_MEV[0]:.2f} MeV",
#     "{}^{22}Na " + f"{NA22_MEV[1]:.2f} MeV",
#     "{}^{22}Na " + f"{NA22_MEV[2]:.2f} MeV",
#     "{}^{60}Co " + f"{CO60_MEV[0]:.2f} MeV",
#     "{}^{60}Co " + f"{CO60_MEV[1]:.2f} MeV",
#     "{}^{40}K "  + f"{K40_MEV[0]:.2f} MeV",
#     "{}^{208}Tl "+ f"{TL208_MEV[0]:.2f} MeV",
#           ]

if ROOT.gROOT.FindObject('cv'):
    ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 800, 600)
lg = ROOT.TLegend(0.48, 0.61, 0.75, 0.83)

zeros = np.zeros(len(means_A), dtype=np.float64)

res_graph_A = ROOT.TGraph(7, real_peaks_ordered[:-2], resolutions['A'][:-2])
res_graph_A.SetMarkerColor(COLOR_B)
res_graph_A.SetMarkerStyle(20)
# correction_B = 1
# res_graph_B = ROOT.TGraph(8, real_peaks_ordered, resolutions['B'])
# res_graph_B.SetMarkerColor(COLOR_B)
# res_graph_B.SetMarkerStyle(20)

res_graph_A.SetTitle("Energy resolution")
res_graph_A.GetXaxis().SetTitle("Energy (MeV)")
res_graph_A.GetYaxis().SetTitle("Resolution (%)")

lg.AddEntry(res_graph_A, "Detector A")
# lg.AddEntry(res_graph_B, "Detector B")
res_graph_A.Draw("AP")
# res_graph_B.Draw("P same")

# Add labels
latex = ROOT.TLatex()
latex.SetTextSize(0.02)
latex.SetTextAlign(12)  # alineación: horizontal izquierda, vertical centrada

for i in range(len(resolutions['A'])):
    latex.DrawLatex(real_peaks_ordered[i] + 0.05, resolutions['A'][i], labels[i])  # pequeño offset en x


lg.Draw()
cv.Update()
cv.Draw()

In [72]:
res_trend_A = ROOT.TF1('res_trend_A', 'TMath::Sqrt([0]^2 + [1]^2*x + [2]^2*x^2)/x', 0, 3)
res_trend_A.FixParameter(0, 0)
res_graph_A.Fit(res_trend_A, 'RS 0')
# res_trend_B = ROOT.TF1('res_trend_B', 'TMath::Sqrt([0]^2 + [1]^2*x + [2]^2*x^2)/x', 0, 3)
# # res_trend_B.FixParameter(0, 0)
# res_graph_B.Fit(res_trend_B, 'RS 0')

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      7.84277
NDf                       =            5
Edm                       =   7.4847e-08
NCalls                    =          204
p0                        =            0                      	 (fixed)
p1                        =     -19.3451   +/-   0.479702    
p2                        =  -0.00112087   +/-   5.27106     


In [73]:
ROOT.gROOT.FindObject('cv').Close()

cv = ROOT.TCanvas('cv', 'cv', 1200, 800)
lg = ROOT.TLegend(0.65, 0.75, 0.86, 0.88)

lg.AddEntry(res_graph_A, "Detector A")
# lg.AddEntry(res_graph_B, "Detector B")

res_trend_A.SetLineColor(COLOR_A)
res_trend_A.SetLineStyle(2)
res_trend_A.SetLineWidth(2)
# res_trend_B.SetLineColor(COLOR_B)
# res_trend_B.SetLineStyle(2)
# res_trend_B.SetLineWidth(2)

res_graph_A.Draw("AP")
# res_graph_B.Draw("P same")
res_trend_A.Draw("same")
# res_trend_B.Draw("same")
lg.Draw()

# get parameters
fit_pars_res = {}
eqboxes = {}
for counter, chip, fitfn in zip([0,1], ['A', ], [res_trend_A, ]):
    eqboxes[f'{chip}'] = ROOT.TPaveText(0.6, 0.6-counter*(0.13), 0.86, 0.8-counter*(0.13), "NDC")
    box =  eqboxes[f'{chip}']
    box.SetFillColor(0)
    box.SetBorderSize(1)
    a = fitfn.GetParameter(0)
    b = fitfn.GetParameter(1)
    c = fitfn.GetParameter(2)
    sa = fitfn.GetParError(0)
    sb = fitfn.GetParError(1)
    sc = fitfn.GetParError(2)

    fit_pars_res[f'a_{chip}'] = a
    fit_pars_res[f'b_{chip}'] = b
    fit_pars_res[f'c_{chip}'] = c
    fit_pars_res[f'sa_{chip}'] = sa
    fit_pars_res[f'sb_{chip}'] = sb
    fit_pars_res[f'sc_{chip}'] = sc

    box.AddText(f"Detector {chip}")
    box.AddText("#sqrt{a^{2} + b^{2}E + c^{2}E^{2}} / E")
    box.AddText(f"a = {a:.4f} #pm {sa:.4f}")
    box.AddText(f"b = {b:.4f} #pm {sb:.4f}")
    box.AddText(f"c = {c:.4f} #pm {sc:.4f}")
    box.Draw()

# Add labels
latex = ROOT.TLatex()
latex.SetTextSize(0.02)
latex.SetTextAlign(12)  # alineación: horizontal izquierda, vertical centrada

for i in range(len(resolutions['A'])):
    latex.DrawLatex(real_peaks_ordered[i] + 0.05, resolutions['A'][i], labels[i])  # pequeño offset en x


cv.Draw()